In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
import xgboost as xgb
import statsmodels.formula.api as smf
import shap
import warnings

In [2]:

print("正在执行阶段一：读取与清洗面板数据 (V3 最终版)...")

# 1. 解析并提取 CPIH 年度数据
cpih_lines = []
with open('../Data_Clean/data/data_raw/series-020326.csv', 'r') as f:
    for line in f:
        line = line.strip().replace('"', '')
        parts = line.split(',')
        if len(parts) == 2 and parts[0].isdigit():
            cpih_lines.append((int(parts[0]), float(parts[1])))

df_cpih = pd.DataFrame(cpih_lines, columns=['year', 'cpih_rate'])
df_cpih = df_cpih[(df_cpih['year'] >= 2011) & (df_cpih['year'] <= 2025)].sort_values('year').reset_index(drop=True)

# 构建 CPIH 累计通胀指数
df_cpih['cpih_factor'] = 1 + df_cpih['cpih_rate'] / 100
df_cpih['cpih_index'] = 1.0
for i in range(1, len(df_cpih)):
    df_cpih.loc[i, 'cpih_index'] = df_cpih.loc[i-1, 'cpih_index'] * df_cpih.loc[i, 'cpih_factor']

# 2. 读取并处理主面板数据
df = pd.read_csv('../Data_Clean/data/data_cleaned/Master_Panel_Data_Final.csv')
df_lad = df[df['Geo_level'] == 'LAD'].copy()

# 清除充电桩的保密掩码 '[x]'
df_lad['ports_per_10k_pop'] = df_lad['ports_per_10k_pop'].replace('[x]', np.nan)
df_lad['ports_per_10k_pop'] = pd.to_numeric(df_lad['ports_per_10k_pop'], errors='coerce')
df_lad = df_lad.sort_values(by=['ONS_code', 'year'])

# 合并 CPIH，并计算出“真实 GDHI”
df_lad = df_lad.merge(df_cpih[['year', 'cpih_rate', 'cpih_index']], on='year', how='left')
df_lad['real_gdhi'] = df_lad['gdhi_per_head'] / df_lad['cpih_index']

# 3. 向量化经济外推模型
gdhi_2011 = df_lad[df_lad['year'] == 2011].set_index('ONS_code')['real_gdhi']
gdhi_2019 = df_lad[df_lad['year'] == 2019].set_index('ONS_code')['real_gdhi']

# 计算 8年真实复合增长率 (Real CAGR)
real_cagr = (gdhi_2019 / gdhi_2011) ** (1/8) - 1
real_cagr = real_cagr.fillna(0.015)

nom_2023 = df_lad[df_lad['year'] == 2023].set_index('ONS_code')['gdhi_per_head']
cpih_2024 = df_cpih.loc[df_cpih['year'] == 2024, 'cpih_rate'].values[0]
cpih_2025 = df_cpih.loc[df_cpih['year'] == 2025, 'cpih_rate'].values[0]

# 推算 2024 和 2025 年名义GDHI
pred_2024 = nom_2023 * (1 + real_cagr) * (1 + cpih_2024 / 100)
pred_2025 = pred_2024 * (1 + real_cagr) * (1 + cpih_2025 / 100)

df_lad.loc[df_lad['year'] == 2024, 'gdhi_per_head'] = df_lad.loc[df_lad['year'] == 2024, 'ONS_code'].map(pred_2024)
df_lad.loc[df_lad['year'] == 2025, 'gdhi_per_head'] = df_lad.loc[df_lad['year'] == 2025, 'ONS_code'].map(pred_2025)

# 4. 指标同步更新与格式化 (新增逻辑)
# 1) 删除难以精准预测且共线性的 gdhi_index
if 'gdhi_index' in df_lad.columns:
    df_lad = df_lad.drop(columns=['gdhi_index'])

# 2) 根据最新补全的 gdhi_per_head，利用 pct_change() 同步计算全新的年增长率
df_lad['gdhi_growth'] = df_lad.groupby('ONS_code')['gdhi_per_head'].pct_change() * 100

# 3) 将人均GDHI和增长率统一保留1位小数
df_lad['gdhi_per_head'] = df_lad['gdhi_per_head'].round(0)
df_lad['gdhi_growth'] = df_lad['gdhi_growth'].round(1)

# 5. 构建滞后变量 (T-1期) 缓解因果内生性
lag_cols = ['charging_ports', 'ports_per_10k_pop', 'gdhi_per_head']
for col in lag_cols:
    df_lad[f'lag1_{col}'] = df_lad.groupby('ONS_code')[col].shift(1)

# 6. 时间截断与清理输出
df_model = df_lad[df_lad['year'] >= 2019].copy()
df_model = df_model.drop(columns=['cpih_rate', 'cpih_index', 'real_gdhi'])

output_filename = '../Data_Clean/data/data_cleaned/Weight_Stage1_Cleaned_Panel.csv'
df_model.to_csv(output_filename, index=False)
print('完成阶段一')

正在执行阶段一：读取与清洗面板数据 (V3 最终版)...
完成阶段一


In [3]:

print("正在执行阶段二：计量经济学固定效应模型 (LAD & Year Two-way FE)...")

# 1. 加载阶段一清洗完毕的数据
df = pd.read_csv('../Data_Clean/data/data_cleaned/Weight_Stage1_Cleaned_Panel.csv')

# 2. 选取核心特征与目标变量
features = ['lag1_ports_per_10k_pop', 'lag1_gdhi_per_head', 'pop_value']
target = 'ev_penetration'

# 剔除由于构建滞后项（T-1）而自然产生缺失值的 2019 年数据
df_model = df[['ONS_code', 'year', target] + features].dropna().copy()

# 3. Z-score 标准化
def standardize(col):
    return (col - col.mean()) / col.std()

df_model[f'{target}_std'] = standardize(df_model[target])
for v in features:
    df_model[f'{v}_std'] = standardize(df_model[v])

# 4. 构建并运行“双向固定效应模型”
# C(ONS_code): 控制地区不随时间变化的固有属性
# C(year): 控制全国统一的宏观时间趋势
formula = f"{target}_std ~ lag1_ports_per_10k_pop_std + lag1_gdhi_per_head_std + pop_value_std + C(ONS_code) + C(year)"

# 拟合模型，使用针对面板数据异方差稳健的“聚类标准误” (以 ONS_code 聚类)
model_fe = smf.ols(formula=formula, data=df_model).fit(
    cov_type='cluster',
    cov_kwds={'groups': df_model['ONS_code']}
)

# 5. 提取并格式化核心权重指标
params = model_fe.params[[f'{f}_std' for f in features]]
pvals = model_fe.pvalues[[f'{f}_std' for f in features]]
conf_int = model_fe.conf_int().loc[[f'{f}_std' for f in features]]

results_df = pd.DataFrame({
    'Feature_Name': ['Lagged Charging Ports Density (T-1)', 'Lagged GDHI per Head (T-1)', 'Population Size'],
    'Standardized_Weight_(Beta)': params.values.round(4),
    'P_Value': pvals.values.round(4),
    'CI_Lower_95%': conf_int[0].values.round(4),
    'CI_Upper_95%': conf_int[1].values.round(4),
    'Significance': ['***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else 'ns (不显著)' for p in pvals.values]
})

# 导出结果
output_filename = 'Stage2_Fixed_Effects_Weights.csv'
results_df.to_csv(output_filename, index=False)
print(results_df.to_string(index=False))

正在执行阶段二：计量经济学固定效应模型 (LAD & Year Two-way FE)...
                       Feature_Name  Standardized_Weight_(Beta)  P_Value  CI_Lower_95%  CI_Upper_95% Significance
Lagged Charging Ports Density (T-1)                      0.2525   0.0416        0.0096        0.4953           **
         Lagged GDHI per Head (T-1)                     -0.2382   0.2811       -0.6714        0.1950     ns (不显著)
                    Population Size                      2.4670   0.0037        0.8019        4.1321          ***


In [4]:
print("=== 阶段三 (最终融合版)：全面滞后、时空基线与非线性 XGBoost ===")

# 1. 加载数据
df = pd.read_csv('../Data_Clean/data/data_cleaned/Weight_Stage1_Cleaned_Panel.csv')

# 1. 类别特征: 'Region' (保留地理空间记忆)
# 2. 滞后基建: 'lag1_ports_per_10k_pop'
# 3. 滞后经济: 'lag1_gdhi_per_head'
# 4. 人口规模: 'pop_value' (控制城市体量)
# 5. 时间基线: 'year' (吸收时代自然增长红利)
categorical_features = ['Region']
numerical_features = ['year', 'lag1_ports_per_10k_pop', 'lag1_gdhi_per_head', 'pop_value']
features = categorical_features + numerical_features
target = 'ev_penetration'

# 剔除空值 (2019年因为没有2018的历史数据，会自动被干净地剥离)
df_model = df.dropna(subset=[target] + features).copy()

# 强制转换类别型特征 (XGBoost 专有要求)
for col in categorical_features:
    df_model[col] = df_model[col].astype('category')

# 2. OOT (Out-of-Time) 跨时间验证切分
# 测试集 (Test)
test_mask = df_model['year'] >= 2024
X_test = df_model.loc[test_mask, features]
y_test = df_model.loc[test_mask, target]

# 训练集+验证集 (Train & Val): <= 2023
hist_mask = df_model['year'] <= 2023
X_hist = df_model.loc[hist_mask, features]
y_hist = df_model.loc[hist_mask, target]

# 内部 8:2 切分验证集用于早停
X_train, X_val, y_train, y_val = train_test_split(X_hist, y_hist, test_size=0.2, random_state=42)

print(f"数据切分完毕 | 训练集: {len(X_train)} | 验证集: {len(X_val)} | OOT测试集: {len(X_test)}")

# 3. 构建开启了类别支持的 XGBoost 架构
xgb_model = xgb.XGBRegressor(
    max_depth=5,
    learning_rate=0.03,
    n_estimators=1000,
    reg_alpha=0.5,
    reg_lambda=2.0,
    enable_categorical=True,
    tree_method='hist',
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    subsample=0.8,         # 每棵树只随机抽取 80% 的城市样本
    colsample_bytree=0.8 # 每棵树只随机抽取 80% 的特征
)

# 拟合与早停监控
print("\n正在训练模型并寻找最优迭代次数...")
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

# 1. 测算 OOT 测试集 (2024-2025 未知未来) 的 R²
y_pred_test = xgb_model.predict(X_test)
r2_test = r2_score(y_test, y_pred_test)

# 2. 测算 Validation 验证集 (历史随机抽取的20%盲盒) 的 R²
y_pred_val = xgb_model.predict(X_val)
r2_val = r2_score(y_val, y_pred_val)

# 3. 测算全量历史集 (2019-2023) 的 R² (这也是 SHAP 解剖的真实底座)
y_pred_hist = xgb_model.predict(X_hist)
r2_hist = r2_score(y_hist, y_pred_hist)

print("\n=== 模型拟合性能诊断 ===")
print(f"[历史全量集 R²]: {r2_hist:.4f} (代表 SHAP 权重解析的底座可靠度)")
print(f"[随机验证集 R²]: {r2_val:.4f} (代表模型在历史区间内的真实泛化能力)")
print(f"[OOT测试集 R²]: {r2_test:.4f} (代表树模型面对指数级外推时的失效)")

# 5. SHAP 权重解析与可视化
print("\n正在生成 SHAP 非线性阈值图与百分比权重表...")
explainer = shap.TreeExplainer(xgb_model)
# 用历史全量数据解释规律最稳定
shap_values = explainer.shap_values(X_hist)

# 定量绝对权重
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_weights_pct = (mean_abs_shap / mean_abs_shap.sum()) * 100

weights_df = pd.DataFrame({
    'Feature': features,
    'Weight_Percentage (%)': shap_weights_pct
}).sort_values(by='Weight_Percentage (%)', ascending=False)

print(weights_df.round(2).to_string(index=False))

plt.figure(figsize=(10, 6))
ax = sns.barplot(
    x='Weight_Percentage (%)',
    y='Feature',
    data=weights_df,
    palette='viridis',
    hue='Feature',
    legend=False
)

# 动态在每个柱子的右侧打上具体的百分比数字
for p in ax.patches:
    width = p.get_width()
    plt.text(
        width + 0.5,                   # x坐标：在柱子末端向右偏移 0.5
        p.get_y() + p.get_height()/2,  # y坐标：柱子的垂直中心
        f'{width:.1f}%',               # 文本内容：保留1位小数的百分比
        ha='left',                     # 水平左对齐
        va='center',                   # 垂直居中对齐
        fontweight='bold',
        fontsize=11
    )

# 设置图表标题与坐标轴
plt.title('Feature Importance (SHAP Absolute Mean Contribution)', fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Relative Importance / Weight Percentage (%)', fontsize=12)
plt.ylabel('Features', fontsize=12)

# 为了右侧的数字不被遮挡，动态扩大 x 轴的上限
plt.xlim(0, weights_df['Weight_Percentage (%)'].max() + 5)
plt.tight_layout()

# 保存图像
output_fig_name = 'Stage3_SHAP_Feature_Importance.png'
plt.savefig(output_fig_name, dpi=300)
plt.close()

print("\n正在生成 SHAP 全景融合图 (绝对权重 + 蜂群分布)...")

# ========================================================
# 终极融合图表：左侧条形权重图 + 右侧 SHAP 蜂群图
# ========================================================
# 创建一个宽幅画布，左右比例约为 1 : 1.2
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7), gridspec_kw={'width_ratios': [1, 1.2]})

# ----------------- 左半部分：全局特征重要性条形图 -----------------
sns.barplot(
    x='Weight_Percentage (%)',
    y='Feature',
    data=weights_df,
    palette='viridis',
    hue='Feature',
    legend=False,
    ax=ax1
)

# 动态打上百分比数字标签
for p in ax1.patches:
    width = p.get_width()
    ax1.text(
        width + 0.5,
        p.get_y() + p.get_height()/2,
        f'{width:.1f}%',
        ha='left',
        va='center',
        fontweight='bold',
        fontsize=12
    )

ax1.set_title('(A) Feature Importance\n(Absolute Mean SHAP Contribution)', fontsize=15, fontweight='bold', pad=15)
ax1.set_xlabel('Relative Weight Percentage (%)', fontsize=13)
ax1.set_ylabel('Features', fontsize=13)
ax1.set_xlim(0, weights_df['Weight_Percentage (%)'].max() + 5)

# ----------------- 右半部分：SHAP 蜂群分布图 -----------------
# 强行将绘图焦点切换到右侧的 ax2 轴
plt.sca(ax2)

# plot_size=None 极其关键，它阻止了 SHAP 篡改我们设置好的画布比例
shap.summary_plot(shap_values, X_hist, show=False, plot_size=None)

ax2.set_title('(B) SHAP Value Distribution\n(Impact on EV Penetration Output)', fontsize=15, fontweight='bold', pad=15)
ax2.set_xlabel('SHAP Value (Drag/Boost on EV %)', fontsize=13)

# ----------------- 整体布局与保存 -----------------
plt.suptitle('Drivers of EV Penetration: Importance and Directional Impact', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()

# 保存
output_fig_name = 'Stage3_SHAP_Combined_Summary.png'
plt.savefig(output_fig_name, dpi=300, bbox_inches='tight')
plt.close()

print(f"✅ 完美融合！【权重条形图 + SHAP蜂群图】已保存至: {output_fig_name}")

# 1. 蜂群图
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_hist, show=False)
plt.title('SHAP Summary: Drivers of EV Penetration', fontsize=14)
plt.tight_layout()
plt.savefig('Stage3_SHAP_Summary_Final.png', dpi=300)
plt.close()

# 2. 充电桩奇点挖掘
plt.figure(figsize=(8, 6))
shap.dependence_plot('lag1_ports_per_10k_pop', shap_values, X_hist, show=False, interaction_index='lag1_gdhi_per_head')
plt.title('Tipping Point Analysis: Charging Infrastructure', fontsize=14)
plt.tight_layout()
plt.savefig('Stage3_SHAP_Ports_TippingPoint.png', dpi=300)
plt.close()

# 3. 财富门槛挖掘
plt.figure(figsize=(8, 6))
shap.dependence_plot('lag1_gdhi_per_head', shap_values, X_hist, show=False, interaction_index=None)
plt.title('Wealth Threshold Analysis (Lagged GDHI)', fontsize=14)
plt.tight_layout()
plt.savefig('Stage3_SHAP_Wealth_Threshold.png', dpi=300)
plt.close()

print("\n 阶段三一切就绪，图表与权重均已锁定。")
# 6. 附加模块：政府基建绩效审计 (基于 SHAP 局部解释提取红黑榜)
print("\n=== 附加模块：2023年地方政府基建绩效审计 (局部 SHAP 解析) ===")

# 1. 获取特定特征在 SHAP 矩阵中的列索引
ports_idx = features.index('lag1_ports_per_10k_pop')
gdhi_idx = features.index('lag1_gdhi_per_head')

# 2. 复制历史数据底表，将 SHAP 局部贡献值拼接到原始数据旁
df_audit = df_model.loc[hist_mask].copy()

# 提取局部的 SHAP 贡献值
df_audit['SHAP_Ports_Contribution'] = shap_values[:, ports_idx]
df_audit['SHAP_Wealth_Contribution'] = shap_values[:, gdhi_idx]

# 3. 锁定 2023 年进行横向比较与排名
df_2023_audit = df_audit[df_audit['year'] == 2023].copy()

# 确保 ONS_geo 在输出列中
output_cols = [
    'ONS_geo', 'ONS_code', 'Region', 'ev_penetration',
    'lag1_ports_per_10k_pop', 'lag1_gdhi_per_head',
    'SHAP_Ports_Contribution', 'SHAP_Wealth_Contribution'
]
# 过滤存在的列，防止因列名微小差异报错
actual_output_cols = [col for col in output_cols if col in df_2023_audit.columns]
df_2023_audit = df_2023_audit[actual_output_cols]

#  制作红黑榜数据
# 红榜 (Top 10)：基建 SHAP 贡献值极高的地区
red_list = df_2023_audit.sort_values(by='SHAP_Ports_Contribution', ascending=False).head(10)

# 黑榜 (Bottom 10)：基建 SHAP 贡献值为极负的地区
black_list = df_2023_audit.sort_values(by='SHAP_Ports_Contribution', ascending=True).head(10)

print("\n 英国 2023 年地方政府【充电基建绩效红榜】Top 10 ")
print(red_list[['ONS_geo', 'lag1_ports_per_10k_pop', 'SHAP_Ports_Contribution']].to_string(index=False))

print("\n 英国 2023 年地方政府【充电基建黑榜】Bottom 10 ")
print(black_list[['ONS_geo', 'lag1_ports_per_10k_pop', 'SHAP_Ports_Contribution']].to_string(index=False))

# 保存审计榜单为 CSV
audit_filename = 'Stage3_Gov_Performance_Audit_2023.csv'
df_2023_audit.sort_values(by='SHAP_Ports_Contribution', ascending=False).to_csv(audit_filename, index=False)


# 图表绘制
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# 左图：红榜 (正向贡献)
sns.barplot(
    x='SHAP_Ports_Contribution',
    y='ONS_geo',
    data=red_list,
    ax=ax1,
    palette='Reds_r',
    hue='ONS_geo',
    legend=False
)
ax1.set_title('Top 10: Infrastructure Outperformers\n(Positive SHAP Contribution to EV %)', fontsize=14, fontweight='bold', color='darkred')
ax1.set_xlabel('SHAP Value (Boost to EV Penetration)', fontsize=12)
ax1.set_ylabel('')
ax1.grid(axis='x', linestyle='--', alpha=0.6)

# 右图：黑榜 (负向拖累)
sns.barplot(
    x='SHAP_Ports_Contribution',
    y='ONS_geo',
    data=black_list,
    ax=ax2,
    palette='Blues_d',
    hue='ONS_geo',
    legend=False
)
ax2.set_title('Bottom 10: Infrastructure Laggards\n(Negative SHAP Drag on EV %)', fontsize=14, fontweight='bold', color='darkblue')
ax2.set_xlabel('SHAP Value (Drag on EV Penetration)', fontsize=12)
ax2.set_ylabel('')
ax2.grid(axis='x', linestyle='--', alpha=0.6)

# 调整布局并保存
plt.suptitle('Local Government Proactivity Index: Charging Infrastructure Impact (2023)', fontsize=18, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.95]) # 为主标题留出空间
plt.savefig('Stage3_Gov_Performance_Audit_Chart.png', dpi=300, bbox_inches='tight')
plt.close()

print(f"\n 'Stage3_Gov_Performance_Audit_Chart.png'")

=== 阶段三 (最终融合版)：全面滞后、时空基线与非线性 XGBoost ===
数据切分完毕 | 训练集: 1139 | 验证集: 285 | OOT测试集: 722

正在训练模型并寻找最优迭代次数...

=== 模型拟合性能诊断 ===
[历史全量集 R²]: 0.8697 (代表 SHAP 权重解析的底座可靠度)
[随机验证集 R²]: 0.5631 (代表模型在历史区间内的真实泛化能力)
[OOT测试集 R²]: 0.2328 (代表树模型面对指数级外推时的失效)

正在生成 SHAP 非线性阈值图与百分比权重表...

--- 机器学习最终版：特征重要性绝对权重 ---
               Feature  Weight_Percentage (%)
                  year              24.059999
lag1_ports_per_10k_pop              21.290001
    lag1_gdhi_per_head              19.809999
             pop_value              19.370001
                Region              15.460000

正在生成 SHAP 全景融合图 (绝对权重 + 蜂群分布)...
✅ 完美融合！【权重条形图 + SHAP蜂群图】已保存至: Stage3_SHAP_Combined_Summary.png

 阶段三一切就绪，图表与权重均已锁定。

=== 附加模块：2023年地方政府基建绩效审计 (局部 SHAP 解析) ===

 英国 2023 年地方政府【充电基建绩效红榜】Top 10 
               ONS_geo  lag1_ports_per_10k_pop  SHAP_Ports_Contribution
           Westminster                   553.6                23.465870
Kensington and Chelsea                   416.9                 9.995850
        City of L

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>